# PDF Analyzer — Notebook (Ollama)

Análisis de documentos PDF usando modelos locales a través de **Ollama**.  
No requiere `transformers`, `torch` ni GPU.  
Entorno conda: **chatbot-pdf**

```bash
conda activate chatbot-pdf
pip install pymupdf requests flask flask-cors
```

Asegúrate de que Ollama está corriendo:
```bash
ollama serve          # en otra terminal
ollama pull qwen3:0.6b
```

In [ ]:
# ── Celda 1: Imports y configuración ────────────────────────────────────────
import os, json, time
import requests
import fitz  # PyMuPDF

# Lee config.json del directorio raíz del proyecto
CONFIG_PATH = os.path.abspath(os.path.join(os.getcwd(), '..', 'config.json'))
if not os.path.exists(CONFIG_PATH):
    CONFIG_PATH = os.path.join(os.getcwd(), 'config.json')

with open(CONFIG_PATH) as f:
    config = json.load(f)

MODEL_NAME  = config['model_name']
OLLAMA_URL  = config.get('ollama_url', 'http://localhost:11434')
MAX_TOKENS  = config.get('max_tokens', 1024)
TEMPERATURE = config.get('temperature', 0.7)

print(f'Config cargado desde : {CONFIG_PATH}')
print(f'Modelo               : {MODEL_NAME}')
print(f'Ollama URL           : {OLLAMA_URL}')

# Verificar Ollama
try:
    r = requests.get(OLLAMA_URL + '/api/tags', timeout=5)
    modelos = [m['name'] for m in r.json().get('models', [])]
    print(f'Ollama OK — {len(modelos)} modelos disponibles: {modelos}')
except Exception as e:
    print(f'[WARN] Ollama no responde: {e}')

In [ ]:
# ── Celda 2: Prompt maestro configurable ────────────────────────────────────
master_prompt = """\
Eres un analizador experto de documentos. Tu tarea es revisar el texto \
del PDF enviado e identificar las siguientes secciones:

1. OBJETIVOS: Identifica si el documento tiene una sección de objetivos \
(puede llamarse: objetivo, propósito, finalidad, alcance, metas).
Si la tiene, extrae su contenido. Si no, indica que no se encontró \
y sugiere dónde podría incluirse.

2. CONCLUSIONES: Identifica si el documento tiene conclusiones \
(puede llamarse: conclusiones, resultados, hallazgos, cierre, resumen final).
Si las tiene, extrae su contenido. Si no, da tus observaciones sobre \
el documento y sugiere qué conclusiones podría incluir.

Al final, da una valoración general del documento de 1 a 5 estrellas \
según su estructura y completitud.\
"""

print(f'Prompt maestro listo ({len(master_prompt)} caracteres).')
print('Edita la variable master_prompt antes de analizar para personalizar.')

In [ ]:
# ── Celda 3: Función analyze_pdf ─────────────────────────────────────────────

def _read_config() -> dict:
    with open(CONFIG_PATH) as f:
        return json.load(f)


def extract_text(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    text = '\n'.join(p.get_text() for p in doc)
    doc.close()
    return text


def analyze_pdf(pdf_path: str, prompt: str = None, verbose: bool = True) -> dict:
    """
    Analiza un PDF local llamando directamente a Ollama.

    Parámetros:
        pdf_path : ruta al archivo PDF
        prompt   : prompt maestro (usa master_prompt si no se especifica)
        verbose  : imprime el resultado en pantalla

    Retorna dict con: texto_extraido, analisis, modelo_usado, tiempo_s
    """
    cfg = _read_config()  # siempre lee la config actualizada
    system = prompt or master_prompt

    if verbose:
        print(f'Extrayendo texto de: {pdf_path}')
    texto = extract_text(pdf_path)

    if verbose:
        print(f'Texto: {len(texto)} caracteres | Llamando a Ollama ({cfg["model_name"]})…')

    payload = {
        'model': cfg['model_name'],
        'messages': [
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': texto[:6000]},
        ],
        'stream': False,
        'options': {
            'num_predict': cfg.get('max_tokens', 1024),
            'temperature': cfg.get('temperature', 0.7),
        },
    }

    t0 = time.time()
    r = requests.post(
        cfg['ollama_url'].rstrip('/') + '/api/chat',
        json=payload, timeout=300,
    )
    r.raise_for_status()
    elapsed = time.time() - t0

    analisis = r.json()['message']['content']

    if verbose:
        print(f'Generado en {elapsed:.1f}s')
        print('─' * 60)
        print(analisis)

    return {
        'texto_extraido': texto,
        'analisis':       analisis,
        'modelo_usado':   cfg['model_name'],
        'tiempo_s':       round(elapsed, 2),
    }


print('analyze_pdf listo. Uso: result = analyze_pdf("/ruta/doc.pdf")')

In [ ]:
# ── Celda 4: Cambiar de modelo ───────────────────────────────────────────────

def change_model(nombre_modelo: str, descargar_si_falta: bool = False) -> None:
    """
    Cambia el modelo activo actualizando config.json.
    No requiere reiniciar el kernel ni el servidor Flask.

    Parámetros:
        nombre_modelo      : nombre del modelo en Ollama (ej: 'qwen3:1.7b')
        descargar_si_falta : si True, ejecuta 'ollama pull' automáticamente
    """
    import subprocess
    cfg = _read_config()
    ollama_base = cfg.get('ollama_url', 'http://localhost:11434')

    # Verifica Ollama
    try:
        r = requests.get(ollama_base + '/api/tags', timeout=5)
        disponibles = [m['name'] for m in r.json().get('models', [])]
    except Exception as e:
        print(f'[ERROR] Ollama no responde: {e}')
        return

    # Verifica si el modelo está disponible
    if nombre_modelo not in disponibles:
        print(f'[WARN] "{nombre_modelo}" no está en Ollama.')
        print(f'       Disponibles: {disponibles}')
        if descargar_si_falta:
            print(f'Descargando {nombre_modelo}…')
            subprocess.run(['ollama', 'pull', nombre_modelo], check=True)
        else:
            print(f'       Ejecuta: ollama pull {nombre_modelo}')
            return

    anterior = cfg['model_name']
    cfg['model_name'] = nombre_modelo
    with open(CONFIG_PATH, 'w') as f:
        json.dump(cfg, f, indent=2)

    print(f'Modelo cambiado: {anterior}  →  {nombre_modelo}')
    print('config.json actualizado. El cambio es inmediato (Flask y notebook).')


print('change_model listo.')
print('Uso: change_model("qwen3:1.7b")')
print('     change_model("llama3:8b", descargar_si_falta=True)')

In [ ]:
# ── Celda 5: Lanzar Flask en background ─────────────────────────────────────
import threading, subprocess, sys as _sys

def start_server(port: int = 5050):
    """
    Lanza server/app.py en un hilo de fondo desde el notebook.
    El servidor leerá config.json en cada request (sin reiniciar).
    """
    server_path = os.path.join(os.path.dirname(CONFIG_PATH), 'server', 'app.py')

    def _run():
        subprocess.run([_sys.executable, server_path])

    t = threading.Thread(target=_run, daemon=True)
    t.start()
    print(f'Servidor Flask iniciado  →  http://localhost:{port}')
    print('Interfaz web            →  interface/index.html')
    return t


# Descomenta para arrancar:
# server_thread = start_server()

In [ ]:
# ── Celda 6: Ejemplo de uso y benchmark comparativo ──────────────────────────

# ─── Ejemplo básico ──────────────────────────────────────────────────────────
PDF_PATH = '/ruta/a/tu/documento.pdf'  # <- cambia esta ruta

if os.path.exists(PDF_PATH):
    resultado = analyze_pdf(PDF_PATH)
    print(f'\nModelo: {resultado["modelo_usado"]} | Tiempo: {resultado["tiempo_s"]}s')
else:
    print(f'PDF no encontrado: {PDF_PATH}')
    print('Actualiza PDF_PATH para probar el análisis.')


# ─── Benchmark entre modelos ─────────────────────────────────────────────────
# Lista de modelos a comparar (deben estar descargados en Ollama)
MODELOS = [
    # 'qwen3:0.6b',
    # 'qwen3:1.7b',
    # 'llama3:8b',
]

if os.path.exists(PDF_PATH) and MODELOS:
    print('\n=== BENCHMARK COMPARATIVO ===')
    resultados = []

    for modelo in MODELOS:
        print(f'\nAnalizando con {modelo}…')
        change_model(modelo)
        r = analyze_pdf(PDF_PATH, verbose=False)
        palabras = len(r['analisis'].split())
        resultados.append({'modelo': modelo, 'tiempo_s': r['tiempo_s'], 'palabras': palabras})
        print(f'  {modelo:<30}  {r["tiempo_s"]:>6}s   ~{palabras} palabras')

    print('\n── Resumen ──────────────────────────────────────────────')
    for row in sorted(resultados, key=lambda x: x['tiempo_s']):
        bar = '█' * min(40, int(row['tiempo_s']))
        print(f"  {row['modelo']:<30} {row['tiempo_s']:>6}s  {bar}")
else:
    print('\nPara el benchmark: descomenta MODELOS y asegúrate de que PDF_PATH existe.')